# Optional raw-data transcription

This notebook demonstrates how to transcribe a raw process-log CSV file and a raw network PCAP file into the formats consumed by the quick start. The included S3 example produces `S3_att.state.gz` and `S3_att.ipal.gz`.

Run this notebook from the repository root after activating the IPAL environment. The pre-transcribed datasets used by `quick_start.ipynb` are already included, so this workflow is optional.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import gzip
import json
import os
import shlex
import shutil
import subprocess

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "dataset":
    ROOT = ROOT.parent
if not (ROOT / "dataset" / "transcriber").is_dir():
    raise RuntimeError("Start Jupyter from the repository root before running this notebook.")

DATA_DIR = ROOT / "dataset" / "transcriber"
CSV_INPUT = DATA_DIR / "S3_att.csv"
PCAP_INPUT = DATA_DIR / "S3_att.pcap"
ATTACKS_JSON = DATA_DIR / "S3_attacks.json"
RULES_FILE = DATA_DIR / "rules-modbus.py"
STATE_OUTPUT = DATA_DIR / "S3_att.state.gz"
NETWORK_OUTPUT = DATA_DIR / "S3_att.ipal.gz"

# S3 CSV timestamps align with the supplied attack JSON when interpreted as UTC.
# Change this timezone when transcribing a dataset recorded in another timezone.
SOURCE_TIMEZONE = timezone.utc

def tool(name):
    local = ROOT / "ipal-dev" / "bin" / name
    resolved = local if local.exists() else shutil.which(name)
    if not resolved:
        raise FileNotFoundError(f"{name} is unavailable. Set up and activate the IPAL environment first.")
    return str(resolved)

TRANSCRIBER = tool("ipal-transcriber")
ADD_ATTACKS = tool("ipal-add-attacks")

for required in (CSV_INPUT, PCAP_INPUT, ATTACKS_JSON, RULES_FILE):
    if not required.exists():
        raise FileNotFoundError(required)

print(f"CSV input: {CSV_INPUT.relative_to(ROOT)}")
print(f"PCAP input: {PCAP_INPUT.relative_to(ROOT)}")
print(f"Default attack labels: {ATTACKS_JSON.relative_to(ROOT)}")

## 1. Transcribe CSV to state.gz

This follows the state experiment notebooks: remove the unused `HMI.P4.Permissive_On` column, normalize HMI column names, parse timestamps, convert process values to the IPAL state representation, and label timestamps covered by the attack JSON.

In [2]:
TIMESTAMP_FORMATS = (
    "%d/%b/%Y %H:%M:%S",
    "%d/%m/%Y %I:%M:%S %p",
    "%d/%B/%Y %H:%M:%S",
)

def load_attacks(attacks_file):
    if attacks_file is None:
        return []
    with Path(attacks_file).open(encoding="utf-8") as handle:
        attacks = json.load(handle)
    print(f"Loaded {len(attacks)} attack interval(s) from {attacks_file.name}")
    return attacks

def get_attack(timestamp, attacks):
    return next(
        (attack for attack in attacks if attack["start"] <= timestamp <= attack["end"]),
        None,
    )

def parse_timestamp(value, source_timezone=timezone.utc):
    text = str(value).strip()
    for timestamp_format in TIMESTAMP_FORMATS:
        try:
            parsed = datetime.strptime(text, timestamp_format)
            return parsed.replace(tzinfo=source_timezone).timestamp()
        except ValueError:
            continue
    raise ValueError(f"Unsupported timestamp format: {text}")

def normalize_column_name(column):
    parts = str(column).split(".")
    return parts[1] if len(parts) >= 3 else str(column).strip()

def transcribe_csv_to_state(csv_input, state_output, attacks_file=None):
    frame = pd.read_csv(csv_input)
    if len(frame.columns) > 1 and frame.columns[1] == "HMI.P4.Permissive_On":
        frame.drop(columns=[frame.columns[1]], inplace=True)
    frame.columns = [normalize_column_name(column) for column in frame.columns]
    frame["malicious"] = "Normal"

    attacks = load_attacks(attacks_file)
    attributes = list(frame.columns)
    processed = 0
    malicious_count = 0

    with gzip.open(state_output, "wt", encoding="utf-8") as output:
        for row_number, row in enumerate(frame.itertuples(index=False, name=None), 1):
            try:
                timestamp = parse_timestamp(row[0], SOURCE_TIMEZONE)
                state = {
                    attributes[index]: round(float(row[index]), 9)
                    for index in range(1, len(row) - 1)
                }
            except (TypeError, ValueError) as error:
                raise ValueError(f"Failed to transcribe CSV row {row_number}: {error}") from error

            attack = get_attack(timestamp, attacks)
            malicious = attack["id"] if attack is not None else False
            malicious_count += int(attack is not None)
            output.write(json.dumps({
                "timestamp": timestamp,
                "state": state,
                "malicious": malicious,
            }) + "\n")
            processed += 1

    print(f"Created {state_output.name}: {processed} records, {malicious_count} malicious records")
    return Path(state_output)

In [ ]:
state_file = transcribe_csv_to_state(CSV_INPUT, STATE_OUTPUT, ATTACKS_JSON)

## 2. Transcribe PCAP to ipal.gz

The equivalent shell command is:

```bash
ipal-transcriber --pcap dataset/transcriber/S3_att.pcap \
    --protocols modbus \
    --ipal.output dataset/transcriber/S3_att.ipal.gz \
    --rules dataset/transcriber/rules-modbus.py
```

In [ ]:
transcribe_command = [
    TRANSCRIBER,
    "--pcap", str(PCAP_INPUT),
    "--protocols", "modbus",
    "--ipal.output", str(NETWORK_OUTPUT),
    "--rules", str(RULES_FILE),
]
print("$ " + shlex.join(transcribe_command))
subprocess.run(transcribe_command, cwd=ROOT, check=True)
print(f"Created {NETWORK_OUTPUT.name}")

## 3. Ensure network timestamp order

PCAP transcription can occasionally produce records that are not strictly ordered by timestamp. The following step detects inversions and performs a stable in-place sort only when necessary.

In [ ]:
def open_json_lines(path, mode="r"):
    path = Path(path)
    if path.suffix == ".gz":
        return gzip.open(path, mode + "t", encoding="utf-8")
    return path.open(mode, encoding="utf-8")

def _get_ipal_timestamp(record):
    timestamp = record.get("timestamp", record.get("ts", record.get("time")))
    if timestamp is None:
        raise KeyError("Record does not contain a timestamp field.")
    return float(timestamp)

def _stable_sort_ipal_records(records):
    ordered_records = []
    inversion_count = 0
    first_inversion = None
    previous_timestamp = None

    for line_index, record in enumerate(records, 1):
        timestamp = _get_ipal_timestamp(record)
        if previous_timestamp is not None and timestamp < previous_timestamp:
            inversion_count += 1
            if first_inversion is None:
                first_inversion = (line_index, previous_timestamp, timestamp)
        previous_timestamp = timestamp
        ordered_records.append((timestamp, line_index, record))

    if inversion_count == 0:
        return records, 0, None

    ordered_records.sort(key=lambda item: (item[0], item[1]))
    return [record for _, _, record in ordered_records], inversion_count, first_inversion

def _ensure_ipal_timestamp_order(path):
    path = Path(path)
    records = []
    with open_json_lines(path) as input_file:
        for line_number, line in enumerate(input_file, 1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as error:
                    raise ValueError(f"Invalid JSON at {path}:{line_number}") from error

    normalized, inversion_count, first_inversion = _stable_sort_ipal_records(records)
    if inversion_count == 0:
        print(f"Timestamp order is already valid: {path.name}")
        return path

    temporary_path = path.with_name(path.name.replace(".ipal.gz", ".ordered.ipal.gz"))
    with open_json_lines(temporary_path, "w") as output_file:
        for record in normalized:
            output_file.write(json.dumps(record) + "\n")
    temporary_path.replace(path)

    line_number, previous, current = first_inversion
    print(f"Normalized {inversion_count} inversion(s); first at line {line_number}: {previous} -> {current}")
    return path

In [ ]:
network_file = _ensure_ipal_timestamp_order(NETWORK_OUTPUT)

## 4. Add attack labels

Attack labeling is intentionally a separate step. By default it uses `dataset/transcriber/S3_attacks.json`. To transcribe another attack dataset, set `ATTACKS_JSON` to its real attack JSON before running this cell. Set it to `None` only for benign data.

In [ ]:
def add_attacks(dataset_files, attacks_file):
    if attacks_file is None:
        print("No attack JSON selected; skipping attack labeling.")
        return
    attacks_file = Path(attacks_file)
    if not attacks_file.exists():
        raise FileNotFoundError(attacks_file)

    for dataset_file in map(Path, dataset_files):
        temporary_output = dataset_file.with_name(
            dataset_file.name.replace(".gz", ".labeled.gz")
        )
        command = [
            ADD_ATTACKS,
            "--attacks", str(attacks_file),
            "--input", str(dataset_file),
            "--output", str(temporary_output),
        ]
        print("$ " + shlex.join(command))
        subprocess.run(command, cwd=ROOT, check=True)
        temporary_output.replace(dataset_file)
        print(f"Added attack labels to {dataset_file}")

add_attacks([state_file, network_file], ATTACKS_JSON)

## Outputs

The generated files are:

- `dataset/transcriber/S3_att.state.gz`
- `dataset/transcriber/S3_att.ipal.gz`